# Fase 4 v2 — 00: template de landmarks + harness de métricas

Fundación del rediseño de homografía robusta multi-feature. Aquí:
1. Render del template oficial del campo + **landmarks nombrados** (sanity visual).
2. Definición del **set de clips de evaluación** (cenital + lateral).
3. **Harness de métricas** (`reproj_cm`, `jitter_cm`, `coverage`) corrido sobre un
   solver baseline (camino C `solve()`), end-to-end, para validar la tubería de medición.

Spec: `docs/specs/2026-06-16-homografia-robusta-multifeature.md`.

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import cv2
import decord

# repo raiz: sube desde el cwd hasta encontrar src/core (portable a cualquier clon)
REPO = Path.cwd().resolve()
while not (REPO / 'src/core/field_template.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.core import field_template as ft
from src.core import field_landmarks as fl
from src.core import homography_metrics as hm
from src.core import auto_homography as ah

OUT = REPO / 'notebooks/fase_4_homografia/v2_robust/outputs'
OUT.mkdir(parents=True, exist_ok=True)
print('REPO:', REPO)
print('landmarks:', len(fl.LANDMARK_POINTS), '| circulo:', fl.CENTER_CIRCLE)

## 1. Template + landmarks (sanity visual)

In [2]:
canvas, world_to_px = ft.render_field(scale=2.6, margin_cm=10.0)
canvas = fl.draw_landmarks(canvas.copy(), world_to_px, radius=4)
# etiquetas
for name, pt in fl.LANDMARK_POINTS.items():
    x, y = world_to_px(pt)
    cv2.putText(canvas, name, (x + 5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.32, (255, 255, 255), 1, cv2.LINE_AA)
cv2.imwrite(str(OUT / 'template_landmarks.png'), cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR))
print('template ->', OUT / 'template_landmarks.png', '| size', canvas.shape)
print('puntos:', list(fl.LANDMARK_POINTS.keys()))

template -> /workspace/FutBotMX-UAQTeam/notebooks/fase_4_homografia/v2_robust/outputs/template_landmarks.png | size (525, 684, 3)
puntos: ['inner_tl', 'inner_tr', 'inner_br', 'inner_bl', 'center_top', 'center_bot', 'center', 'circle_top', 'circle_bot', 'circle_left', 'circle_right', 'penL_top', 'penL_bot', 'penR_top', 'penR_bot', 'goalL_top', 'goalL_bot', 'goalR_top', 'goalR_bot', 'postY_top', 'postY_bot', 'postB_top', 'postB_bot']


## 2. Set de clips de evaluación (cenital + lateral)

Subsamplea ~`N_FRAMES` frames repartidos en cada video. Cenital = cámara superior
(donde el camino C funciona). Lateral = cámara externa (donde debe mejorar).

In [3]:
MG = REPO / 'data/raw'
if not MG.exists():
    MG = Path('/workspace/Meta_Glasses')

EVAL_CLIPS = {
    # nombre -> (ruta, vista)
    'cenital_9933': (Path('/workspace/Meta_Glasses/18abril/Camara_superior/IMG_9933.MOV'), 'cenital'),
    'cenital_9938': (Path('/workspace/Meta_Glasses/18abril/Camara_superior/IMG_9938.MOV'), 'cenital'),
    'lateral_9913': (Path('/workspace/Meta_Glasses/18abril/Cámaras/IMG_9913.MOV'), 'lateral'),
    'lateral_9914': (Path('/workspace/Meta_Glasses/18abril/Cámaras/IMG_9914.MOV'), 'lateral'),
    'lateral_9915': (Path('/workspace/Meta_Glasses/18abril/Cámaras/IMG_9915.MOV'), 'lateral'),
}
N_FRAMES = 24

def load_clip(path, n):
    vr = decord.VideoReader(str(path))
    idx = np.unique(np.linspace(0, len(vr) - 1, n).round().astype(int))
    return [vr[i].asnumpy() for i in idx]  # RGB

clips = {}
meta = {}
for name, (path, view) in EVAL_CLIPS.items():
    if not path.exists():
        print('FALTA:', name, path); continue
    frames = load_clip(path, N_FRAMES)
    clips[name] = frames
    meta[name] = {'view': view, 'n_frames': len(frames), 'res': list(frames[0].shape[:2])}
    print(f'{name:16s} {view:8s} frames={len(frames):3d} res={frames[0].shape[1]}x{frames[0].shape[0]}')
json.dump(meta, open(OUT / 'eval_clips.json', 'w'), indent=2)

cenital_9933     cenital  frames= 24 res=1080x1920


cenital_9938     cenital  frames= 24 res=1080x1920


lateral_9913     lateral  frames= 24 res=1920x1080


lateral_9914     lateral  frames= 24 res=1920x1080


lateral_9915     lateral  frames= 24 res=1920x1080


## 3. Harness de métricas con solver baseline (camino C)

`solve()` es HSV puro (sin red), tuneado para cenital. Auto-etiqueta las 4 esquinas
del rectángulo interior contra el template (por cercanía en cm, sin asumir orden).

Nota honesta: el camino C ajusta `H` exactamente a esas 4 esquinas (DLT), así que su
`reproj_cm` sobre ellas es ~0 por construcción; **el `jitter_cm` y la `coverage` son
las señales informativas** de este baseline. El `reproj` sobre landmarks independientes
(círculo, áreas) llega cuando el solver multi-feature los detecte (nb 02–03).

In [4]:
INNER = ['inner_tl', 'inner_tr', 'inner_br', 'inner_bl']
INNER_CM = np.array([fl.LANDMARK_POINTS[n] for n in INNER], dtype=np.float64)
TOL_CM = 45.0

def baseline_solver(frame_rgb, idx):
    bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
    try:
        res = ah.solve(bgr)
    except Exception:
        return hm.FrameResult(idx, None)
    H = getattr(res, 'H', None)
    if H is None or not getattr(res, 'ok', False) or getattr(res, 'corners', None) is None:
        return hm.FrameResult(idx, None)
    corners = np.asarray(res.corners, dtype=np.float64).reshape(-1, 2)
    cm = hm.project_img_to_world(H, corners)
    det = {}
    for i, c in enumerate(cm):
        j = int(np.argmin(np.linalg.norm(INNER_CM - c, axis=1)))
        if np.linalg.norm(INNER_CM[j] - c) < TOL_CM:
            det[INNER[j]] = (float(corners[i, 0]), float(corners[i, 1]))
    return hm.FrameResult(idx, np.asarray(H, dtype=np.float64), det)

res = hm.run_variant(clips, baseline_solver)
print('=== BASELINE (camino C) ===')
print(f"global: reproj_cm={res['reproj_cm']}  jitter_cm={res['jitter_cm']}  coverage={res['coverage']:.2f}  n={res['n_frames']}")
print(f"{'clip':16s} {'view':8s} {'cov':>5s} {'reproj':>8s} {'jitter':>8s}")
for clip, d in res['per_clip'].items():
    rp = '-' if d['reproj_cm'] is None else f"{d['reproj_cm']:.2f}"
    jt = '-' if d['jitter_cm'] is None else f"{d['jitter_cm']:.2f}"
    print(f"{clip:16s} {meta[clip]['view']:8s} {d['coverage']:5.2f} {rp:>8s} {jt:>8s}")

=== BASELINE (camino C) ===
global: reproj_cm=6.629106678701342e-14  jitter_cm=8.280988631160356e-14  coverage=0.41  n=120
clip             view       cov   reproj   jitter
cenital_9933     cenital   1.00     0.00     0.00
cenital_9938     cenital   0.50     0.00     0.00
lateral_9913     lateral   0.46     0.00     0.00
lateral_9914     lateral   0.04     0.00        -
lateral_9915     lateral   0.04     0.00        -


In [5]:
# Persistir resultado del baseline (consumible por nb 01+ para comparar)
save = {k: v for k, v in res.items() if k != 'per_clip_frames'}
json.dump(save, open(OUT / 'baseline_pathC_metrics.json', 'w'), indent=2)
print('guardado ->', OUT / 'baseline_pathC_metrics.json')
print(json.dumps(save, indent=2)[:600])

guardado -> /workspace/FutBotMX-UAQTeam/notebooks/fase_4_homografia/v2_robust/outputs/baseline_pathC_metrics.json
{
  "reproj_cm": 6.629106678701342e-14,
  "jitter_cm": 8.280988631160356e-14,
  "coverage": 0.4083333333333333,
  "n_frames": 120,
  "per_clip": {
    "cenital_9933": {
      "n_frames": 24,
      "coverage": 1.0,
      "reproj_cm": 5.489360521347991e-14,
      "jitter_cm": 8.280988631160356e-14
    },
    "cenital_9938": {
      "n_frames": 24,
      "coverage": 0.5,
      "reproj_cm": 5.987992797920323e-14,
      "jitter_cm": 6.689947352745379e-14
    },
    "lateral_9913": {
      "n_frames": 24,
      "coverage": 0.4583333333333333,
      "reproj_cm": 1.1546319456101628e-13,
      "jitter_


## Resumen

- `field_landmarks.py` + `homography_metrics.py` validados end-to-end sobre frames reales.
- Template + landmarks renderizados (`outputs/template_landmarks.png`).
- Set de clips eval fijo (`outputs/eval_clips.json`).
- Baseline camino C medido (`outputs/baseline_pathC_metrics.json`) — referencia para nb 01+.

Siguiente: `01` baseline detallado cenital vs lateral; `02` solver multi-feature.